In [0]:
from pyspark.sql import functions as F, Window

B = lambda n: spark.table(f"workspace.bronze.{n}")

def latest(df, keys, *order_cols):
    """Keep one row per key: the most recent version."""
    w = Window.partitionBy(*keys).orderBy(*[F.col(c).desc() for c in order_cols])
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def save(df, name, key):
    full = f"workspace.silver.{name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full))
    t = spark.table(full)
    n, d = t.count(), t.select(key).distinct().count()
    assert n == d, f"{name}: {n - d} duplicate {key} values"   # fail the job if the key isn't unique
    print(f"{name}: {n} rows, key '{key}' unique")

In [0]:
stores = B("stores").select(
    "store_id", F.trim("store_name").alias("store_name"),
    F.trim("city").alias("city"), F.trim("state").alias("state"),
    F.col("opened_date").cast("date").alias("opened_date"),
    "_source_file", "_ingested_at")
save(latest(stores, ["store_id"], "_source_file"), "stores", "store_id")

customers = B("customers").select(
    "customer_id",
    F.trim("first_name").alias("first_name"), F.trim("last_name").alias("last_name"),
    F.lower(F.trim("email")).alias("email"), "phone",
    F.trim("city").alias("city"),
    F.expr("try_cast(created_at as timestamp)").alias("created_at"),
    F.expr("try_cast(updated_at as timestamp)").alias("updated_at"),
    "_source_file", "_ingested_at")
save(latest(customers, ["customer_id"], "updated_at", "_source_file"), "customers", "customer_id")

stores: 10 rows, key 'store_id' unique
customers: 200 rows, key 'customer_id' unique


In [0]:
orders = B("orders").select(
    "order_id", "customer_id", "store_id",
    F.expr("try_cast(order_date as timestamp)").alias("order_ts"),
    F.trim("payment_mode").alias("payment_mode"),
    F.trim("order_status").alias("order_status"),
    F.expr("try_cast(updated_at as timestamp)").alias("updated_at"),
    "_source_file", "_ingested_at"
).withColumn("order_date", F.to_date("order_ts"))
save(latest(orders, ["order_id"], "updated_at", "_source_file"), "orders", "order_id")

items = B("order_items").select(
    "order_item_id", "order_id", "product_id",
    F.col("quantity").cast("int").alias("quantity"),
    F.col("unit_price").cast("decimal(10,2)").alias("unit_price"),
    "_source_file", "_ingested_at"
).withColumn("line_total", F.col("quantity") * F.col("unit_price"))
save(latest(items, ["order_item_id"], "_source_file"), "order_items", "order_item_id")

orders: 4000 rows, key 'order_id' unique
order_items: 12001 rows, key 'order_item_id' unique


In [0]:
products = B("products").select(
    F.col("id").alias("product_id"), F.trim("title").alias("title"),
    F.trim("category").alias("category"), "description",
    F.col("price").cast("decimal(10,2)").alias("price"),
    F.col("rating.rate").alias("rating_rate"),
    F.col("rating.count").alias("rating_count"),
    "image", "_source_file", "_ingested_at")
save(latest(products, ["product_id"], "_source_file"), "products", "product_id")

products: 20 rows, key 'product_id' unique


In [0]:
o, i = spark.table("workspace.silver.orders"), spark.table("workspace.silver.order_items")
assert o.filter("order_ts IS NULL").count() == 0, "orders with unparseable date"
assert i.join(o, "order_id", "left_anti").count() == 0, "order_items without an order"
print("Silver core: all checks passed")

Silver core: all checks passed
